In [75]:
import numpy as np
import pandas as pd

np.random.seed(1234)
import xgboost as xgb


In [76]:
train = pd.read_csv("../data/santander-customer-satisfaction/train.csv")
test =  pd.read_csv("../data/santander-customer-satisfaction/test.csv")

print("Length:", len(train))
train.head()

Length: 76020


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [77]:
train["TARGET"].value_counts()

# TARGET = 0 : 73,012개  → 약 96%
# TARGET = 1 :  3,008개  → 약 4%

# 레이블 불균형 -> ROC-AUC, Recall, Precision, F1

TARGET
0    73012
1     3008
Name: count, dtype: int64

In [78]:
train.shape 
#→ 76,020행, 370개 컬럼

print(train.describe())

                  ID           var3         var15  imp_ent_var16_ult1  \
count   76020.000000   76020.000000  76020.000000        76020.000000   
mean    75964.050723   -1523.199277     33.212865           86.208265   
std     43781.947379   39033.462364     12.956486         1614.757313   
min         1.000000 -999999.000000      5.000000            0.000000   
25%     38104.750000       2.000000     23.000000            0.000000   
50%     76043.000000       2.000000     28.000000            0.000000   
75%    113748.750000       2.000000     40.000000            0.000000   
max    151838.000000     238.000000    105.000000       210000.000000   

       imp_op_var39_comer_ult1  imp_op_var39_comer_ult3  \
count             76020.000000             76020.000000   
mean                 72.363067               119.529632   
std                 339.315831               546.266294   
min                   0.000000                 0.000000   
25%                   0.000000                 

In [79]:
# ID 와  Target 분리
# 모델 학습에 필요하지 않은 ID 를 제거
# 정답값 Target을 feature 데이터에서 분리한다

# train ID 제거
train = train.drop(columns=["ID"])

# test ID는 나중에 제출할 때 필요하므로 따로 저장
test_id = test["ID"].copy()
test = test.drop(columns=["ID"])

# TARGET 따로 저장
train_y = train["TARGET"].copy()

# feature에서 TARGET 제거
train = train.drop(columns=["TARGET"])

In [80]:
print(train.shape)
print(test.shape)
print(train_y.shape) #정답 target

# TARGET = 0 → 만족 고객
# TARGET = 1 → 불만족 고객

(76020, 369)
(75818, 369)
(76020,)


In [81]:
# var3 이상치(-999999) 처리
# var3는 국적 코드로 추정되는 변수인데, 결측치가 -999999로 인코딩되어 있음
# (describe()에서 min이 -999999, mean이 -1523인데 25/50/75%는 전부 2인 것에서 확인 가능)
# -999999를 그대로 두면 스케일이 왜곡되고 XGBoost가 이 값을 실제 큰 음수로 취급하게 됨
# 최빈값인 2로 대체 (public kernel들에서 널리 쓰이는 처리 방식)

print("var3 == -999999 개수 (train):", (train["var3"] == -999999).sum())
print("var3 == -999999 개수 (test):", (test["var3"] == -999999).sum())
print("var3 최빈값:", train["var3"].mode()[0])

train["var3"] = train["var3"].replace(-999999, 2)
test["var3"] = test["var3"].replace(-999999, 2)

print("\n처리 후 var3 min (train):", train["var3"].min())
print("처리 후 var3 min (test):", test["var3"].min())

var3 == -999999 개수 (train): 116
var3 == -999999 개수 (test): 120
var3 최빈값: 2

처리 후 var3 min (train): 0
처리 후 var3 min (test): 0


In [82]:
#각 행에서 0의 개수 세기
#전처리 -> Feature Engineering 
# 각 행(row)마다 값이 0인 feature가 몇 개인지 세어서 n0라는 새로운 컬럼을 추가하는 거야.
# n0를 만드는 이유: 이건 0인 feature를 제거하는 게 아니라, 각 고객(row)이 몇 개의 feature에서 0을 가지고 있는지를 새로운 정보로 만드는 거야.

train["n0"] = (train == 0).sum(axis=1)
test["n0"] = (test == 0).sum(axis=1)
train[["n0"]].head()

C:\Users\mega\AppData\Local\Temp\ipykernel_1148\1220695049.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["n0"] = (train == 0).sum(axis=1)
C:\Users\mega\AppData\Local\Temp\ipykernel_1148\1220695049.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["n0"] = (test == 0).sum(axis=1)


,n0
0,355
1,329
2,340
3,309
4,319


In [83]:
#값이 하나뿐인 Constant Feature 제거
# 모든 row들에서 값이 동일한 contant feature 제거
# 모든 사람에게 동일한 값만 가지고 있어서 예측에 도움이 안되는 컬럼 34개 삭제

print("## Removing constant features")

constant_features = [
    col for col in train.columns
    if train[col].nunique(dropna=False) == 1
]

print("Number of constant features:", len(constant_features))

train = train.drop(columns=constant_features)
test = test.drop(columns=constant_features)

## Removing constant features
Number of constant features: 34


In [84]:
#제거된 column만 보려면
constant_features[:20]

['ind_var2_0',
 'ind_var2',
 'ind_var27_0',
 'ind_var28_0',
 'ind_var28',
 'ind_var27',
 'ind_var41',
 'ind_var46_0',
 'ind_var46',
 'num_var27_0',
 'num_var28_0',
 'num_var28',
 'num_var27',
 'num_var41',
 'num_var46_0',
 'num_var46',
 'saldo_var28',
 'saldo_var27',
 'saldo_var41',
 'saldo_var46']

In [85]:
# 완전히 동일한 Feature 제거
# Identical feature 제거

from itertools import combinations

to_remove = []

for f1, f2 in combinations(train.columns, 2):

    if f1 not in to_remove and f2 not in to_remove:

        if train[f1].equals(train[f2]):
            to_remove.append(f2)

print("Number of identical features removed:", len(to_remove))

Number of identical features removed: 29


In [86]:
# 수치형 변수 변환(로그 변환, Log Transformation)
#var38 로그 변환
# var38 값이 너무 큰 범위를 가지고 있어서 로그를 씌워 분포를 압축하는 전처리

train["var38"] = np.log(train["var38"])
test["var38"] = np.log(test["var38"])

train["var38"].describe()

count    76020.000000
mean        11.482248
std          0.560589
min          8.549418
25%         11.125358
50%         11.575047
75%         11.684828
max         16.908131
Name: var38, dtype: float64

In [87]:
# identical feature 제거 전 test 상태를 보관
tc = test.copy()  # 먼저 원본 보관

train = train.drop(columns=to_remove)
test = test.drop(columns=to_remove)

In [88]:
# Test 값을 Train의 min/max 범위 안으로 제한
print("Setting min-max limits on test data")

for col in train.columns:

    train_min = train[col].min()
    train_max = train[col].max()

    test[col] = test[col].clip(
        lower=train_min,
        upper=train_max
    )

Setting min-max limits on test data


In [89]:
# XGBoost 파라미터
# (아직 모델을 XGBoost로 확정한 게 아니라서 지금은 적용 안 함 -
#  모델 비교 끝나고 XGBoost가 최종 선택되면 이 값들을 XGBClassifier에 반영할 예정)

params = {
    "objective": "binary:logistic",
    "booster": "gbtree",
    "eval_metric": "auc",
    "eta": 0.0202048,
    "max_depth": 5,
    "subsample": 0.6815,
    "colsample_bytree": 0.701,
    "seed": 1234
}

params

{'objective': 'binary:logistic',
 'booster': 'gbtree',
 'eval_metric': 'auc',
 'eta': 0.0202048,
 'max_depth': 5,
 'subsample': 0.6815,
 'colsample_bytree': 0.701,
 'seed': 1234}

In [90]:
#전처리된 train을 X, y 로 준비

# train       # 전처리가 끝난 feature
# train_y     # TARGET
# test        # Kaggle test.csv 전처리 완료
# test_id     # Kaggle test의 ID

In [91]:
# ============================================
# 1. Feature / Target 준비
# ============================================
 
X_features = train.copy()   #고객정보
y_labels = train_y.copy()   # TARGET 정답

print("X shape:", X_features.shape)
print("y shape:", y_labels.shape)

print("\nTARGET 분포:")
print(y_labels.value_counts())
print("\nTARGET 비율:")
print(y_labels.value_counts(normalize=True))

X shape: (76020, 307)
y shape: (76020,)

TARGET 분포:
TARGET
0    73012
1     3008
Name: count, dtype: int64

TARGET 비율:
TARGET
0    0.960431
1    0.039569
Name: proportion, dtype: float64


In [92]:
#학습/테스트 데이터 분리, 분포 확인
from sklearn.model_selection import train_test_split




X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels,
                                                    test_size=0.2, random_state=0, stratify=y_labels)
train_cnt = y_train.count()
test_cnt = y_test.count()
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))




print(' 학습 세트 레이블 값 분포 비율')
print(y_train.value_counts()/train_cnt)
print('\n 테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts()/test_cnt)


학습 세트 Shape:(60816, 307), 테스트 세트 Shape:(15204, 307)
 학습 세트 레이블 값 분포 비율
TARGET
0    0.960438
1    0.039562
Name: count, dtype: float64

 테스트 세트 레이블 값 분포 비율
TARGET
0    0.960405
1    0.039595
Name: count, dtype: float64


In [93]:
#  X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                                    test_size=0.3, random_state=0, stratify=y_train)


In [94]:
# XGB모델 학습 AUC 점수 확인
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score




# n_estimators는 500으로, learning_rate 0.05, random state는 예제 수행 시마다 동일 예측 결과를 위해 설정.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=0.05, early_stopping_rounds=100, eval_metric='auc',random_state=156)


In [95]:


# 성능 평가 지표를 auc로, 조기 중단 파라미터는 100으로 설정하고 학습 수행.
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])




# 모델 비교(어떤 모델을 튜닝할지 결정) 단계이므로 X_test가 아닌 X_val로 평가
# -> X_test는 맨 마지막, 최종 튜닝된 모델 평가에서 딱 한 번만 사용
xgb_roc_score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
print('ROC AUC (Validation): {0:.4f}'.format(xgb_roc_score))


[0]	validation_0-auc:0.84533	validation_1-auc:0.82264
[1]	validation_0-auc:0.85296	validation_1-auc:0.82895
[2]	validation_0-auc:0.85349	validation_1-auc:0.82858
[3]	validation_0-auc:0.85571	validation_1-auc:0.83482
[4]	validation_0-auc:0.85741	validation_1-auc:0.83567
[5]	validation_0-auc:0.85777	validation_1-auc:0.83682
[6]	validation_0-auc:0.85802	validation_1-auc:0.83728
[7]	validation_0-auc:0.85844	validation_1-auc:0.83731
[8]	validation_0-auc:0.85891	validation_1-auc:0.83713
[9]	validation_0-auc:0.86162	validation_1-auc:0.83886
[10]	validation_0-auc:0.86335	validation_1-auc:0.83967
[11]	validation_0-auc:0.86485	validation_1-auc:0.84009
[12]	validation_0-auc:0.86558	validation_1-auc:0.83996
[13]	validation_0-auc:0.86649	validation_1-auc:0.84116
[14]	validation_0-auc:0.86736	validation_1-auc:0.84134
[15]	validation_0-auc:0.86804	validation_1-auc:0.84108
[16]	validation_0-auc:0.86848	validation_1-auc:0.84046
[17]	validation_0-auc:0.86957	validation_1-auc:0.84111
[18]	validation_0-au

In [96]:
# LightGBM 모델 학습 AUC 점수 확인
# XGBoost와 동일한 조건(n_estimators=500, learning_rate=0.05, early stopping 100)으로 비교
from lightgbm import LGBMClassifier, early_stopping

lgbm_clf = LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=156, verbose=-1)

In [97]:
lgbm_clf.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=100)]
)

# 모델 비교(어떤 모델을 튜닝할지 결정) 단계이므로 X_test가 아닌 X_val로 평가
# -> X_test는 맨 마지막, 최종 튜닝된 모델 평가에서 딱 한 번만 사용
lgbm_roc_score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
print('LightGBM ROC AUC (Validation): {0:.4f}'.format(lgbm_roc_score))

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[63]	training's auc: 0.901583	training's binary_logloss: 0.117481	valid_1's auc: 0.844743	valid_1's binary_logloss: 0.133628
LightGBM ROC AUC (Validation): 0.8447


In [98]:
# Logistic Regression 모델 학습 AUC 점수 확인
# 트리 기반 모델(XGBoost, LightGBM)과 달리 스케일에 민감하므로 표준화 필요
# 스케일러는 X_tr로만 학습(fit)하고 X_val/X_test에는 transform만 적용 -> 데이터 누수 방지
#
# 결과: ROC AUC 0.7727 - 트리 모델(XGBoost 0.8219, LightGBM 0.8238) 대비 크게 낮아
# 이후 튜닝 대상에서는 제외. 코드는 비교 기록용으로 남겨두고 주석 처리.

# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_tr_scaled = scaler.fit_transform(X_tr)
# X_val_scaled = scaler.transform(X_val)
# X_test_scaled = scaler.transform(X_test)

# # TARGET 불균형(96:4)을 고려해 class_weight='balanced' 사용
# lr_clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=156)

In [99]:
# lr_clf.fit(X_tr_scaled, y_tr)

# lr_roc_score = roc_auc_score(y_val, lr_clf.predict_proba(X_val_scaled)[:, 1])
# print('Logistic Regression ROC AUC (Validation): {0:.4f}'.format(lr_roc_score))

In [100]:
# ============================================
# 모델 비교 (동일한 전처리, 동일한 X_tr/X_val split 기준 - X_test는 아직 미사용)
# ============================================

print(f"XGBoost              ROC AUC (Val): {xgb_roc_score:.4f}")
print(f"LightGBM             ROC AUC (Val): {lgbm_roc_score:.4f}")
print("Logistic Regression  ROC AUC (Val): 0.7727  (참고용 - 위 셀에서 주석 처리됨)")

XGBoost              ROC AUC (Val): 0.8443
LightGBM             ROC AUC (Val): 0.8447
Logistic Regression  ROC AUC (Val): 0.7727  (참고용 - 위 셀에서 주석 처리됨)


In [101]:
# ============================================
# LightGBM 하이퍼파라미터 튜닝 (StratifiedKFold + RandomizedSearchCV)
# 중요: X_test/y_test는 최종 평가 전까지 절대 사용하지 않음 -> 탐색은 X_train/y_train만 사용
# ============================================

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform

param_distributions = {
    'num_leaves': randint(15, 128),
    'max_depth': [-1, 4, 5, 6, 7, 8],
    'learning_rate': uniform(0.01, 0.09),     # 0.01 ~ 0.10
    'n_estimators': randint(200, 800),
    'min_child_samples': randint(10, 100),
    'subsample': uniform(0.6, 0.4),           # 0.6 ~ 1.0
    'colsample_bytree': uniform(0.5, 0.5),    # 0.5 ~ 1.0
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(0, 2),
}

# CPU 코어를 탐색(n_jobs=-1)에 몰아주기 위해 개별 모델은 단일 스레드로 고정
# subsample_freq: LightGBM은 bagging_freq(=subsample_freq)가 0(기본값)이면
# subsample 값을 아예 무시하고 bagging을 수행하지 않음 -> subsample을 실제로 적용하려면
# subsample_freq를 반드시 1 이상으로 지정해야 함
base_lgbm = LGBMClassifier(random_state=156, verbose=-1, n_jobs=1, subsample_freq=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=156)

random_search = RandomizedSearchCV(
    estimator=base_lgbm,
    param_distributions=param_distributions,
    n_iter=40,
    scoring='roc_auc',
    cv=cv,
    random_state=156,
    n_jobs=-1,
    verbose=1
)

In [102]:
# X_train/y_train(80% 학습 세트, X_test 미포함)에 대해서만 5-fold CV 탐색 수행
# 40 조합 x 5 fold = 200회 학습 -> 시간이 다소 걸릴 수 있음
random_search.fit(X_train, y_train)

print("Best CV ROC AUC:", random_search.best_score_)
print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV ROC AUC: 0.8446032206054008
Best params:
  colsample_bytree: 0.7440534221615848
  learning_rate: 0.028239182801244885
  max_depth: 4
  min_child_samples: 78
  n_estimators: 370
  num_leaves: 113
  reg_alpha: 0.9453972714793342
  reg_lambda: 1.604626287129203
  subsample: 0.9233860881823783


In [103]:
# ============================================
# 어떤 하이퍼파라미터가 점수에 가장 큰 영향을 줬는지 분석
# 40번의 시도(cv_results_)를 이용해 각 파라미터와 mean_test_score의 상관관계 확인
# (X_train 내부 CV 결과만 사용 -> X_test는 여전히 사용 안 함)
# ============================================

import pandas as pd

cv_results = pd.DataFrame(random_search.cv_results_)

# 연속형 파라미터: mean_test_score와의 상관계수
numeric_params = [
    'param_num_leaves', 'param_learning_rate', 'param_n_estimators',
    'param_min_child_samples', 'param_subsample', 'param_colsample_bytree',
    'param_reg_alpha', 'param_reg_lambda'
]

corr_df = cv_results[numeric_params + ['mean_test_score']].astype(float).corr()['mean_test_score'].drop('mean_test_score')
corr_df = corr_df.sort_values(key=abs, ascending=False)

print("=== 파라미터별 mean_test_score와의 상관관계 (절대값 큰 순) ===")
print(corr_df)

# 범주형 파라미터(max_depth)는 상관계수 대신 그룹별 평균 점수로 확인
print("\n=== max_depth별 평균 mean_test_score ===")
print(cv_results.groupby('param_max_depth')['mean_test_score'].mean().sort_values(ascending=False))

=== 파라미터별 mean_test_score와의 상관관계 (절대값 큰 순) ===
param_learning_rate       -0.794922
param_n_estimators        -0.418674
param_num_leaves          -0.146301
param_colsample_bytree     0.142632
param_reg_alpha           -0.057252
param_subsample           -0.035179
param_reg_lambda           0.018405
param_min_child_samples    0.016692
Name: mean_test_score, dtype: float64

=== max_depth별 평균 mean_test_score ===
param_max_depth
 8    0.839451
 5    0.839197
 4    0.838056
 6    0.837555
 7    0.831420
-1    0.814629
Name: mean_test_score, dtype: float64


In [104]:
# ============================================
# 트리 개수(n_estimators)를 CV 기반으로 안정적으게 추정
# X_tr/X_val 한 번의 split에 의존하는 대신, X_train을 5-fold(탐색 때와 동일한 cv)로 나눠서
# 폴드마다 early stopping -> best_iteration을 구하고 평균냄
# (X_test는 여전히 사용 안 함)
# ============================================

best_params_no_n = {k: v for k, v in random_search.best_params_.items() if k != 'n_estimators'}

best_iterations = []

for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_fold_tr, X_fold_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_fold_tr, y_fold_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    fold_model = LGBMClassifier(**best_params_no_n, n_estimators=2000, random_state=156, verbose=-1, subsample_freq=1)
    fold_model.fit(
        X_fold_tr, y_fold_tr,
        eval_set=[(X_fold_val, y_fold_val)],
        eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=100)]
    )
    best_iterations.append(fold_model.best_iteration_)
    print(f"Fold {fold_idx + 1} best_iteration: {fold_model.best_iteration_}")

avg_best_iteration = int(round(sum(best_iterations) / len(best_iterations)))
print(f"\n5-fold 평균 best_iteration: {avg_best_iteration}")

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[310]	valid_0's auc: 0.842943	valid_0's binary_logloss: 0.1325
Fold 1 best_iteration: 310


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[331]	valid_0's auc: 0.840905	valid_0's binary_logloss: 0.134881
Fold 2 best_iteration: 331


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[310]	valid_0's auc: 0.85366	valid_0's binary_logloss: 0.13047
Fold 3 best_iteration: 310


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[221]	valid_0's auc: 0.838392	valid_0's binary_logloss: 0.134161
Fold 4 best_iteration: 221


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[250]	valid_0's auc: 0.848147	valid_0's binary_logloss: 0.132637
Fold 5 best_iteration: 250

5-fold 평균 best_iteration: 284


In [105]:
# ============================================
# 평균 트리 개수로 X_train 전체(80%)를 사용해 최종 모델 학습
# 이전 방식(X_tr 70%만 사용)과 달리 X_train 전체를 학습에 활용
# 트리 개수를 이미 알고 있으므로 early stopping 불필요 (X_test는 아직 사용 안 함)
# ============================================

final_lgbm = LGBMClassifier(
    **best_params_no_n,
    n_estimators=avg_best_iteration,
    random_state=156,
    verbose=-1,
    subsample_freq=1
)

final_lgbm.fit(X_train, y_train)

print(f"최종 모델 트리 개수: {avg_best_iteration}")
print(f"학습에 사용한 데이터: X_train 전체 {X_train.shape[0]}행 (이전: X_tr {int(X_train.shape[0]*0.7)}행)")

최종 모델 트리 개수: 284
학습에 사용한 데이터: X_train 전체 60816행 (이전: X_tr 42571행)


In [106]:
# ============================================
# 최종 ROC-AUC 평가 - X_test/y_test를 final_lgbm에 대해서만, 처음이자 유일하게 사용
# ============================================

tuned_lgbm_roc_score = roc_auc_score(y_test, final_lgbm.predict_proba(X_test)[:, 1])

# 참고: lgbm_roc_score는 모델 선택 단계에서 X_val로 잰 값이라 데이터셋이 달라
# 아래 X_test 점수와 직접 비교하면 안 됨 (참고용으로만 같이 표시)
print(f"LightGBM (default, 모델 선택 단계 참고값) ROC AUC (Val):  {lgbm_roc_score:.4f}")
print(f"LightGBM (tuned, 최종 모델)               ROC AUC (Test): {tuned_lgbm_roc_score:.4f}")

LightGBM (default, 모델 선택 단계 참고값) ROC AUC (Val):  0.8447
LightGBM (tuned, 최종 모델)               ROC AUC (Test): 0.8262


In [107]:
# ============================================
# 튜닝된 LightGBM 모델을 pickle로 저장
# -> 이후 다른 모델(XGBoost 튜닝, PCA 실험 등)을 돌리다가 커널을 껐다 켜거나
#    변수를 덮어써도 이 시점의 final_lgbm 결과를 잃어버리지 않도록 디스크에 보관
# ============================================

import pickle
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

model_path = model_dir / "final_lgbm_tuned.pkl"

with open(model_path, "wb") as f:
    pickle.dump(
        {
            "model": final_lgbm,
            "params": best_params_no_n,
            "n_estimators": avg_best_iteration,
            "test_roc_auc": tuned_lgbm_roc_score,
            "feature_names": list(X_train.columns),
        },
        f,
    )

print(f"저장 완료: {model_path.resolve()}")

# 나중에 불러올 때:
# with open("models/final_lgbm_tuned.pkl", "rb") as f:
#     saved = pickle.load(f)
# final_lgbm = saved["model"]

저장 완료: C:\big21\ml-data-analysis\01_classification_santander\models\final_lgbm_tuned.pkl


In [108]:
# ============================================
# PCA 실험 0/3: 공정 비교를 위한 대조군
# final_lgbm은 X_train 전체(X_val 포함)로 학습됐기 때문에 X_val로 채점하면 누수(leak)
# -> 0-importance 제거/PCA 없이, 같은 튜닝된 하이퍼파라미터를 X_tr에만 학습 + X_val로 조기종료/채점
#    (final_lgbm은 그대로 보류 - 건드리지 않음)
# ============================================

control_lgbm = LGBMClassifier(
    **best_params_no_n,
    n_estimators=2000,
    random_state=156,
    verbose=-1,
    subsample_freq=1
)

control_lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=100)]
)

control_val_roc_score = roc_auc_score(y_val, control_lgbm.predict_proba(X_val)[:, 1])
print(f"대조군 (튜닝된 파라미터, PCA/필터링 없음) X_val ROC AUC: {control_val_roc_score:.4f}")

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[315]	valid_0's auc: 0.848371	valid_0's binary_logloss: 0.13237
대조군 (튜닝된 파라미터, PCA/필터링 없음) X_val ROC AUC: 0.8484


In [109]:
# ============================================
# PCA 실험 1/3: 0-importance feature 제거
# final_lgbm(X_train 전체로 학습된 기존 튜닝 모델)의 feature_importances_ 기준으로
# 중요도가 0인 컬럼을 찾아서 제거 (모델 재학습 없이 이미 학습된 final_lgbm만 조회)
# ============================================

importances = pd.Series(final_lgbm.feature_importances_, index=X_train.columns)
zero_importance_cols = importances[importances == 0].index.tolist()

print(f"0-importance feature 개수: {len(zero_importance_cols)} / {len(importances)}")
print(zero_importance_cols)

X_tr_f = X_tr.drop(columns=zero_importance_cols)
X_val_f = X_val.drop(columns=zero_importance_cols)
X_test_f = X_test.drop(columns=zero_importance_cols)

print("\n필터링 후 shape (X_tr_f):", X_tr_f.shape)

0-importance feature 개수: 202 / 307
['imp_op_var40_comer_ult1', 'imp_op_var40_comer_ult3', 'imp_op_var40_efect_ult1', 'imp_op_var40_efect_ult3', 'imp_op_var40_ult1', 'imp_sal_var16_ult1', 'ind_var1', 'ind_var6_0', 'ind_var6', 'ind_var8', 'ind_var12', 'ind_var13_corto_0', 'ind_var13_corto', 'ind_var13_largo_0', 'ind_var13_largo', 'ind_var13_medio_0', 'ind_var14_0', 'ind_var14', 'ind_var17_0', 'ind_var17', 'ind_var18_0', 'ind_var19', 'ind_var20_0', 'ind_var20', 'ind_var24_0', 'ind_var24', 'ind_var25_cte', 'ind_var26_0', 'ind_var26_cte', 'ind_var25_0', 'ind_var31_0', 'ind_var31', 'ind_var32_cte', 'ind_var32_0', 'ind_var33_0', 'ind_var33', 'ind_var34_0', 'ind_var37_cte', 'ind_var37_0', 'ind_var40_0', 'ind_var40', 'ind_var44_0', 'ind_var44', 'num_var1_0', 'num_var1', 'num_var6_0', 'num_var6', 'num_var8', 'num_var12', 'num_var13_0', 'num_var13_corto_0', 'num_var13_corto', 'num_var13_largo_0', 'num_var13_largo', 'num_var13_medio_0', 'num_var14_0', 'num_var14', 'num_var17_0', 'num_var17', 'num_

In [110]:
# ============================================
# PCA 실험 2/3: PCA 5개 컴포넌트 추가 (팀원들 방식과 동일 - 원래 feature는 유지하고 추가)
# 스케일링 + PCA는 X_tr_f로만 fit -> X_val_f/X_test_f는 transform만 적용 (누수 방지)
# ============================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_tr_scaled = scaler_pca.fit_transform(X_tr_f)
X_val_scaled = scaler_pca.transform(X_val_f)
X_test_scaled = scaler_pca.transform(X_test_f)

pca = PCA(n_components=5, random_state=156)
pca_tr = pca.fit_transform(X_tr_scaled)
pca_val = pca.transform(X_val_scaled)
pca_test = pca.transform(X_test_scaled)

pca_cols = [f"pca_{i+1}" for i in range(5)]

X_tr_pca = X_tr_f.copy()
X_val_pca = X_val_f.copy()
X_test_pca = X_test_f.copy()

X_tr_pca[pca_cols] = pca_tr
X_val_pca[pca_cols] = pca_val
X_test_pca[pca_cols] = pca_test

print("PCA 5개 설명 분산 비율:", pca.explained_variance_ratio_.round(4))
print("누적 설명 분산:", pca.explained_variance_ratio_.sum().round(4))
print("최종 shape (X_tr_pca):", X_tr_pca.shape)

PCA 5개 설명 분산 비율: [0.1941 0.1024 0.0617 0.0562 0.0498]
누적 설명 분산: 0.4642
최종 shape (X_tr_pca): (42571, 110)


In [111]:
# ============================================
# PCA 실험 3/3: 0-importance 제거 + PCA 5개 추가된 feature로 재학습, X_val로 채점
# (final_lgbm과 동일하게 X_train 전체로 재학습하지 않고, X_tr/X_val 구조를 유지해서
#  대조군(control_val_roc_score)과 공정하게 비교되도록 함 -> X_test는 여전히 사용 안 함)
# ============================================

pca_lgbm = LGBMClassifier(
    **best_params_no_n,
    n_estimators=2000,
    random_state=156,
    verbose=-1,
    subsample_freq=1
)

pca_lgbm.fit(
    X_tr_pca, y_tr,
    eval_set=[(X_val_pca, y_val)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=100)]
)

pca_val_roc_score = roc_auc_score(y_val, pca_lgbm.predict_proba(X_val_pca)[:, 1])

print("=== X_val 기준 비교 (전부 같은 튜닝된 하이퍼파라미터, X_tr에만 학습) ===")
print(f"대조군 (원본 feature, PCA/필터링 없음)        ROC AUC: {control_val_roc_score:.4f}")
print(f"0-importance 제거 + PCA 5개 추가              ROC AUC: {pca_val_roc_score:.4f}")

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[228]	valid_0's auc: 0.850381	valid_0's binary_logloss: 0.13199
=== X_val 기준 비교 (전부 같은 튜닝된 하이퍼파라미터, X_tr에만 학습) ===
대조군 (원본 feature, PCA/필터링 없음)        ROC AUC: 0.8484
0-importance 제거 + PCA 5개 추가              ROC AUC: 0.8504


In [112]:
# ============================================
# PCA 최종화 1/3: 0-importance 제거 + PCA 5개를 X_train 전체(80%) 기준으로 다시 fit
# (탐색 때는 X_tr만 썼지만, 최종 모델은 final_lgbm처럼 X_train 전체를 활용하기 위해
#  스케일러/PCA를 X_train 전체로 재적합. X_test는 transform만 - 아직 채점 안 함)
# ============================================

X_train_f = X_train.drop(columns=zero_importance_cols)
X_test_f2 = X_test.drop(columns=zero_importance_cols)

scaler_pca_final = StandardScaler()
X_train_f_scaled = scaler_pca_final.fit_transform(X_train_f)
X_test_f2_scaled = scaler_pca_final.transform(X_test_f2)

pca_final = PCA(n_components=5, random_state=156)
pca_train_final = pca_final.fit_transform(X_train_f_scaled)
pca_test_final = pca_final.transform(X_test_f2_scaled)

X_train_pca_final = X_train_f.copy()
X_test_pca_final = X_test_f2.copy()
X_train_pca_final[pca_cols] = pca_train_final
X_test_pca_final[pca_cols] = pca_test_final

print("PCA 설명 분산 비율 (X_train 기준):", pca_final.explained_variance_ratio_.round(4))
print("shape:", X_train_pca_final.shape, X_test_pca_final.shape)

PCA 설명 분산 비율 (X_train 기준): [0.1929 0.1027 0.0617 0.0559 0.0496]
shape: (60816, 110) (15204, 110)


In [113]:
# ============================================
# PCA 최종화 2/3: CV 기반 n_estimators 재추정 (PCA 추가된 feature 기준)
# (X_test는 여전히 사용 안 함)
# ============================================

best_iterations_pca = []

for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train_pca_final, y_train)):
    X_fold_tr, X_fold_val = X_train_pca_final.iloc[tr_idx], X_train_pca_final.iloc[val_idx]
    y_fold_tr, y_fold_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    fold_model = LGBMClassifier(**best_params_no_n, n_estimators=2000, random_state=156, verbose=-1, subsample_freq=1)
    fold_model.fit(
        X_fold_tr, y_fold_tr,
        eval_set=[(X_fold_val, y_fold_val)],
        eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=100)]
    )
    best_iterations_pca.append(fold_model.best_iteration_)
    print(f"Fold {fold_idx + 1} best_iteration: {fold_model.best_iteration_}")

avg_best_iteration_pca = int(round(sum(best_iterations_pca) / len(best_iterations_pca)))
print(f"\n5-fold 평균 best_iteration (PCA): {avg_best_iteration_pca}")

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[416]	valid_0's auc: 0.841405	valid_0's binary_logloss: 0.132891
Fold 1 best_iteration: 416


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[391]	valid_0's auc: 0.841926	valid_0's binary_logloss: 0.134553
Fold 2 best_iteration: 391


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[416]	valid_0's auc: 0.853073	valid_0's binary_logloss: 0.130671
Fold 3 best_iteration: 416


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[191]	valid_0's auc: 0.837076	valid_0's binary_logloss: 0.134346
Fold 4 best_iteration: 191


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[313]	valid_0's auc: 0.847904	valid_0's binary_logloss: 0.132673
Fold 5 best_iteration: 313

5-fold 평균 best_iteration (PCA): 345


In [114]:
# ============================================
# PCA 최종화 3/3: X_train 전체로 최종 학습 -> X_test로 딱 한 번 채점
# (X_test는 이 셀에서 final_lgbm_pca에 대해서만, 처음이자 유일하게 사용)
# ============================================

final_lgbm_pca = LGBMClassifier(
    **best_params_no_n,
    n_estimators=avg_best_iteration_pca,
    random_state=156,
    verbose=-1,
    subsample_freq=1
)

final_lgbm_pca.fit(X_train_pca_final, y_train)

tuned_lgbm_pca_roc_score = roc_auc_score(y_test, final_lgbm_pca.predict_proba(X_test_pca_final)[:, 1])

print(f"기존 튜닝 LightGBM (PCA 전)              X_test ROC AUC: {tuned_lgbm_roc_score:.4f}")
print(f"0-importance 제거 + PCA 5개 추가         X_test ROC AUC: {tuned_lgbm_pca_roc_score:.4f}")

기존 튜닝 LightGBM (PCA 전)              X_test ROC AUC: 0.8262
0-importance 제거 + PCA 5개 추가         X_test ROC AUC: 0.8272


In [115]:
# ============================================
# PCA 최종 모델도 pickle로 저장 (기존 final_lgbm_tuned.pkl은 그대로 유지)
# ============================================

pca_model_path = model_dir / "final_lgbm_pca_tuned.pkl"

with open(pca_model_path, "wb") as f:
    pickle.dump(
        {
            "model": final_lgbm_pca,
            "params": best_params_no_n,
            "n_estimators": avg_best_iteration_pca,
            "test_roc_auc": tuned_lgbm_pca_roc_score,
            "zero_importance_cols_removed": zero_importance_cols,
            "scaler": scaler_pca_final,
            "pca": pca_final,
            "feature_names": list(X_train_pca_final.columns),
        },
        f,
    )

print(f"저장 완료: {pca_model_path.resolve()}")

저장 완료: C:\big21\ml-data-analysis\01_classification_santander\models\final_lgbm_pca_tuned.pkl
